## Model Selection 

In [1]:
import pandas as pd

df = pd.read_csv('../datasets/final/final.csv')
df.head()

,season,holiday,workingday,weathersit,temp,hum,windspeed,cnt,year,month_sin,month_cos,dayofweek_sin,dayofweek_cos
0,1,0,0,2,0.344167,0.805833,0.160446,985,2011,0.5,0.866025,-0.974928,-0.222521
1,1,0,0,2,0.363478,0.696087,0.248539,801,2011,0.5,0.866025,-0.781831,0.623490
2,1,0,1,1,0.196364,0.437273,0.248309,1349,2011,0.5,0.866025,0.000000,1.000000
3,1,0,1,1,0.200000,0.590435,0.160296,1562,2011,0.5,0.866025,0.781831,0.623490
4,1,0,1,1,0.226957,0.436957,0.186900,1600,2011,0.5,0.866025,0.974928,-0.222521


## Random Splitting

i tried training on 2011 and testing on 2012 but it didnt workout 
as there is a growing trend between the years

so we'll stick with random splitting

In [2]:
X = df.drop('cnt', axis=1)
y = df['cnt']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models to test:
1. LR
2. Lasso
3. Ridge
4. DT
5. RF
6. GB

In [3]:
df.head()

,season,holiday,workingday,weathersit,temp,hum,windspeed,cnt,year,month_sin,month_cos,dayofweek_sin,dayofweek_cos
0,1,0,0,2,0.344167,0.805833,0.160446,985,2011,0.5,0.866025,-0.974928,-0.222521
1,1,0,0,2,0.363478,0.696087,0.248539,801,2011,0.5,0.866025,-0.781831,0.623490
2,1,0,1,1,0.196364,0.437273,0.248309,1349,2011,0.5,0.866025,0.000000,1.000000
3,1,0,1,1,0.200000,0.590435,0.160296,1562,2011,0.5,0.866025,0.781831,0.623490
4,1,0,1,1,0.226957,0.436957,0.186900,1600,2011,0.5,0.866025,0.974928,-0.222521


In [4]:
## feature encoding

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_cols = ['season','weathersit', 'holiday', 'workingday']
num_cols = ['temp'  , 'hum', 'windspeed', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'year']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [5]:
results =[]

In [6]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def evaluate(model):

    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_rmse = mean_squared_error(y_train, train_preds)**0.5
    test_rmse = mean_squared_error(y_test, test_preds)**0.5

    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)

    model_name = model.named_steps['model'].__class__.__name__ if hasattr(model, "named_steps") and "model" in model.named_steps else model.__class__.__name__
    results.append({"model": model_name, "metrics": [train_rmse, test_rmse, train_r2, test_r2]})

    return {
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2
    }

## LR

In [7]:
from sklearn.linear_model import LinearRegression

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

evaluate(lr_pipeline)

{'train_rmse': 810.1303840827106,
 'test_rmse': 802.6160581496324,
 'train_r2': 0.8209588995688877,
 'test_r2': 0.8393487662633551}

observations:
1. very small gap btw train and test metrics
2. slight underfitting but still healthy bias behavious
3. stable generalization

## Ridge

In [8]:
from sklearn.linear_model import Ridge 

ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

evaluate(ridge_pipeline)

{'train_rmse': 810.2177843855296,
 'test_rmse': 803.4423667333319,
 'train_r2': 0.8209202660580311,
 'test_r2': 0.8390178089516699}

observations:
1. no multicollinearity, no regularization needed, data not high dimensional so no need for ridge
2. results say the same, almost indentical to lr

## Lasso

In [9]:
from sklearn.linear_model import Lasso

lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso())
])

evaluate(lasso_pipeline)

{'train_rmse': 810.1955194366332,
 'test_rmse': 803.4681788946082,
 'train_r2': 0.8209301082175515,
 'test_r2': 0.8390074650484889}

observations:
1. no massive sparsity required and signal distributed across features, so lasso not needed
2. results say the same, almost identical to lr

## DT

In [10]:
from sklearn.tree import DecisionTreeRegressor

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])

evaluate(dt_pipeline)

{'train_rmse': 0.0,
 'test_rmse': 956.9143152878011,
 'train_r2': 1.0,
 'test_r2': 0.771642920938161}

observations:

1. classic high variance, perfect memorization but generalization is still decent 
2. overfitting is obvious

## RF

In [11]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])

evaluate(rf_pipeline)

{'train_rmse': 262.33649674900573,
 'test_rmse': 674.7741542825682,
 'train_r2': 0.9812258481484196,
 'test_r2': 0.8864504924286437}

observations:

1. reduced variance and much stronger generalization

## GB

In [12]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor())
])

evaluate(gb_pipeline)

{'train_rmse': 416.8917534833487,
 'test_rmse': 641.6848194191657,
 'train_r2': 0.9525878657082294,
 'test_r2': 0.8973138416942243}

observations:

1. best test performance. lower bias than rf, controlled variance and strong generalization.

# Chosen Model: Gradient Boost

now we'll perform hyperparameter tuning

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
import numpy as np

gb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

param_dist = {
    "model__n_estimators": np.arange(100, 600, 50),
    "model__learning_rate": np.linspace(0.01, 0.2, 20),
    "model__max_depth": [2, 3, 4, 5],
    "model__subsample": [0.6, 0.8, 1.0]
}

In [16]:
random_search = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': array([0.01, ..., 0.19, 0.2 ]), 'model__max_depth': [2, 3, ...], 'model__n_estimators': array([100, 1...50, 500, 550]), 'model__subsample': [0.6, 0.8, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Gui

In [17]:
best_gb = random_search.best_estimator_

print("Best Params:", random_search.best_params_)

evaluate(best_gb)

Best Params: {'model__subsample': 1.0, 'model__n_estimators': np.int64(150), 'model__max_depth': 2, 'model__learning_rate': np.float64(0.14)}


{'train_rmse': 454.40720940187305,
 'test_rmse': 635.1313286180809,
 'train_r2': 0.9436708337579744,
 'test_r2': 0.8994005869597055}